# Stage 4: Main model (LightGBM)

This notebook builds the main shot make model, gradient boosted trees (LightGBM), and compares it with the best
baseline of Stage 3: the spline logistic regression trained on the last season (`logit_w1`, log loss 0.6402).

The validation is the same as in Stage 3: walk-forward by season, test seasons 2015-16 to 2024-25, hold-out season
2025-26 never loaded. Shared code: `xpts.validation`, `xpts.metrics`, `xpts.baselines`, `xpts.gbm`.

## 0. Setup

In [1]:
%load_ext autoreload
# re-import src/xpts after every change, no kernel restart needed
%autoreload 2

import numpy as np
import pandas as pd

from xpts.baselines import make_logit
from xpts.gbm import DEFAULT_PARAMS, GBM_FEATURES, GBMModel
from xpts.metrics import evaluate
from xpts.validation import HOLDOUT_SEASON, walk_forward

df = pd.read_parquet("../data/processed/shots_features.parquet")
df = df[df["SEASON"] != HOLDOUT_SEASON].reset_index(drop=True)  # hold-out season, never used here
y = df["SHOT_MADE_FLAG"]
print(f"shots: {len(df):,}, seasons: {df['SEASON'].min()} to {df['SEASON'].max()}")

shots: 2,305,805, seasons: 2014-15 to 2024-25


## 1. First LightGBM with default parameters

**Model.** `GBMModel` (in `src/xpts/gbm.py`) is a `LGBMClassifier` on the row-wise features of Stage 2:
distance, angle and raw coordinates, the 3PT / corner / heave / overtime flags, period and seconds left,
and `action_group` as a categorical feature. Unlike the logistic regression, the trees find non-linear effects and
interactions (e.g. distance x shot type x angle) by themselves, without splines.

**Parameters** are fixed by hand, not tuned yet (step 3): 300 trees, learning rate 0.05, 31 leaves, at least 200
shots per leaf, 80% of rows and columns per tree. No early stopping, because stopping on the test season would
leak it into the model.

Both models use the 1-season window (the best choice for the logit in Stage 3), so they see exactly the same data.

In [2]:
print(GBM_FEATURES)
print(DEFAULT_PARAMS)

models = {"logit_w1": make_logit, "gbm_w1": GBMModel}
oos = pd.DataFrame({"SEASON": df["SEASON"], "y": y})
rows = []
for name, make in models.items():
    for season, tr, te in walk_forward(df["SEASON"], window=1):
        p = make().fit(df[tr], y[tr]).predict_proba(df[te])[:, 1]
        oos.loc[te, name] = p
        rows.append({"model": name, "season": season, **evaluate(y[te], p)})

results = pd.DataFrame(rows)
oos = oos.dropna(subset=["logit_w1"])  # 2014-15 is only a training season
bias = oos.groupby("SEASON")[list(models)].mean().rsub(oos.groupby("SEASON")["y"].mean(), axis=0)
display(results.groupby("model")[["log_loss", "brier", "auc"]].mean().loc[list(models)].assign(bias=bias.mean()).round(4))
display(results.pivot(index="season", columns="model", values="log_loss")[list(models)].round(4), bias.round(3))

['dist_ft', 'angle_deg', 'LOC_X', 'LOC_Y', 'is_three', 'is_corner3', 'is_heave', 'is_overtime', 'PERIOD', 'secs_left_period', 'action_group']
{'n_estimators': 300, 'learning_rate': 0.05, 'num_leaves': 31, 'min_child_samples': 200, 'subsample': 0.8, 'subsample_freq': 1, 'colsample_bytree': 0.8, 'verbose': -1}


,log_loss,brier,auc,bias
model,,,,
logit_w1,0.6402,0.2255,0.6562,0.0007
gbm_w1,0.6413,0.2260,0.6537,0.0018


model,logit_w1,gbm_w1
season,,
2015-16,0.6367,0.6368
2016-17,0.6386,0.6374
2017-18,0.6478,0.6570
2018-19,0.6372,0.6370
2019-20,0.6367,0.6365
2020-21,0.6446,0.6466
2021-22,0.6395,0.6400
2022-23,0.6401,0.6402
2023-24,0.6406,0.6406


,logit_w1,gbm_w1
SEASON,,
2015-16,-0.012,-0.010
2016-17,0.001,0.002
2017-18,-0.003,0.009
2018-19,-0.006,-0.006
2019-20,0.004,0.004
2020-21,0.016,0.012
2021-22,-0.002,-0.002
2022-23,0.011,0.011
2023-24,0.001,0.001


**Observations.**
- The untuned LightGBM is **not better** than the logistic regression: mean log loss 0.6413 vs 0.6402, AUC 0.654 vs 0.656.
- In 7 of 10 seasons the two are within ±0.001; LightGBM is better in 2016-17 and clearly worse in 2017-18
  (0.657 vs 0.648) and 2020-21. 2017-18 is the season in which the label vocabulary changed (Stage 3, section 5b):
  the trees fitted on 2016-17 split on label patterns that no longer exist.
- So with these features the spline logit already captures most of the signal. Next steps: the training window
  (more seasons give the trees more data, but older labels), then hyperparameter tuning inside the training window.

## 2. Training window: short vs expanding

**Question.** In Stage 3 the logit was best with a 1-season window, because older seasons use the old `ACTION_TYPE`
labels. Trees need more data than a logit, so a longer window could help LightGBM even with the label drift.
We compare windows of 1, 2 and 3 seasons and the expanding window (all previous seasons), same walk-forward test seasons.

In [3]:
windows = {"gbm_w1": 1, "gbm_w2": 2, "gbm_w3": 3, "gbm_exp": None}
rows_w = [r for r in rows if r["model"] in ("logit_w1", "gbm_w1")]
for name, w in windows.items():
    if name == "gbm_w1":
        continue  # already in section 1
    for season, tr, te in walk_forward(df["SEASON"], window=w):
        p = GBMModel().fit(df[tr], y[tr]).predict_proba(df[te])[:, 1]
        oos.loc[df.index[te], name] = p  # oos is shorter than df (2014-15 dropped)
        rows_w.append({"model": name, "season": season, **evaluate(y[te], p)})

res_w = pd.DataFrame(rows_w)
order = ["logit_w1", *windows]
display(res_w.groupby("model")[["log_loss", "brier", "auc"]].mean().loc[order].round(4))
display(res_w.pivot(index="season", columns="model", values="log_loss")[order].round(4))

,log_loss,brier,auc
model,,,
logit_w1,0.6402,0.2255,0.6562
gbm_w1,0.6413,0.2260,0.6537
gbm_w2,0.6412,0.2260,0.6541
gbm_w3,0.6407,0.2257,0.6558
gbm_exp,0.6418,0.2263,0.6539


model,logit_w1,gbm_w1,gbm_w2,gbm_w3,gbm_exp
season,,,,,
2015-16,0.6367,0.6368,0.6368,0.6368,0.6368
2016-17,0.6386,0.6374,0.6379,0.6379,0.6379
2017-18,0.6478,0.6570,0.6586,0.6512,0.6512
2018-19,0.6372,0.6370,0.6378,0.6387,0.6394
2019-20,0.6367,0.6365,0.6360,0.6366,0.6380
2020-21,0.6446,0.6466,0.6454,0.6451,0.6456
2021-22,0.6395,0.6400,0.6396,0.6398,0.6415
2022-23,0.6401,0.6402,0.6398,0.6400,0.6423
2023-24,0.6406,0.6406,0.6402,0.6403,0.6429


**Observations.**
- All windows are within 0.0016 of each other in mean log loss, and none beats the logit (0.6402).
- The mean is driven by 2017-18, the season after the label change: there w3 and the expanding window (identical, both
  train on 2014-15 to 2016-17) lose less (0.6512) than w1 (0.6570), but still more than the logit (0.6478).
- From 2018-19 on, the expanding window is the worst in every season (by 0.001-0.002 in 2022-23 to 2024-25): old seasons
  with the old labels hurt more than the extra data helps.
- In the recent seasons w2 is the best LightGBM and slightly better than the logit (e.g. 0.6398 vs 0.6401 in 2022-23),
  but these differences (~0.0003) are small and need a paired test before we call them real.

## 3a. Paired bootstrap: is any LightGBM really different from the logit?

**Method.** Both models predict the same test shots, so we compare them *paired*: for every shot we take the
difference of log losses (LightGBM − logit) and average it over all test shots of the 10 seasons. Negative = LightGBM
is better. For the 95% CI we resample whole **games** (1000 bootstrap samples), not single shots, because shots in one
game share players, teams and game conditions and are not independent. `share_a_better` is the share of bootstrap samples
in which LightGBM wins. Helper: `paired_bootstrap` in `src/xpts/metrics.py`.

Besides all 10 seasons we show 2018-19 to 2024-25, the seasons after the label change. This split is motivated by
section 2, so it is descriptive, not a separate test.

In [4]:
from xpts.metrics import paired_bootstrap

games = df.loc[oos.index, "GAME_ID"]
after = oos["SEASON"] >= "2018-19"  # seasons after the ACTION_TYPE label change
ci = pd.DataFrame({(m, period): paired_bootstrap(oos.loc[mask, "y"], oos.loc[mask, m], oos.loc[mask, "logit_w1"],
                                                 games[mask])
                   for m in windows
                   for period, mask in [("all", oos["SEASON"].notna()), ("2018-19+", after)]}).T
display(ci.round(5))

# per season for the two candidates for tuning
ci_season = pd.DataFrame({(m, s): paired_bootstrap(g["y"], g[m], g["logit_w1"], games[g.index])
                          for m in ["gbm_w2", "gbm_w3"] for s, g in oos.groupby("SEASON")}).T
display(ci_season[["diff", "ci_low", "ci_high"]].unstack(0).round(5))

diff   ci_low  ci_high  share_a_better
gbm_w1  all       0.00103  0.00091  0.00114           0.000
        2018-19+  0.00029  0.00018  0.00041           0.000
gbm_w2  all       0.00099  0.00087  0.00111           0.000
        2018-19+ -0.00005 -0.00015  0.00005           0.825
gbm_w3  all       0.00043  0.00033  0.00052           0.000
        2018-19+  0.00020  0.00009  0.00033           0.000
gbm_exp all       0.00159  0.00147  0.00170           0.000
        2018-19+  0.00186  0.00170  0.00202           0.000

diff            ci_low           ci_high         
          gbm_w2   gbm_w3   gbm_w2   gbm_w3   gbm_w2   gbm_w3
2015-16  0.00013  0.00013 -0.00021 -0.00021  0.00048  0.00048
2016-17 -0.00070 -0.00070 -0.00103 -0.00103 -0.00038 -0.00038
2017-18  0.01075  0.00338  0.01019  0.00305  0.01129  0.00370
2018-19  0.00058  0.00153  0.00025  0.00116  0.00093  0.00195
2019-20 -0.00072 -0.00007 -0.00101 -0.00038 -0.00042  0.00025
2020-21  0.00072  0.00043  0.00037  0.00010  0.00104  0.00075
2021-22  0.00012  0.00036 -0.00016  0.00004  0.00040  0.00069
2022-23 -0.00031 -0.00008 -0.00056 -0.00036 -0.00007  0.00018
2023-24 -0.00033 -0.00024 -0.00058 -0.00050 -0.00008  0.00001
2024-25 -0.00040 -0.00052 -0.00064 -0.00077 -0.00015 -0.00028

**Observations.**
- Over all 10 seasons every LightGBM is **significantly worse** than the logit (CI above 0): w2 +0.0010 [0.0009; 0.0011],
  w3 +0.0004, expanding window +0.0016. Almost all of it comes from 2017-18 (w2: +0.011).
- After the label change (2018-19+) w2 and the logit are indistinguishable: −0.00005 [−0.00015; 0.00005].
- In the last three seasons (2022-23 to 2024-25) w2 is better by 0.0003-0.0004 with the whole CI below 0: a real but tiny gain.
- The expanding window is the worst in every period, so old labels hurt LightGBM as well. We tune on the 2-season window.

## 3b. Hyperparameter tuning inside the training window

**Rule.** The test season must not influence any choice. For test season S the 2-season window is (S-2, S-1), so we
split it again: train on S-2, validate on S-1, pick the parameters with the best validation log loss and then refit on
both seasons. The number of trees is found by **early stopping** on S-1 (add trees until the validation log loss has not
improved for 50 rounds). The first fold (test 2015-16) has only one training season, so it keeps the default parameters.

**Grid.** `num_leaves` (tree complexity) in {7, 15, 31, 63} x `min_child_samples` (minimum shots per leaf) in
{20, 100, 500}, learning rate 0.05, up to 2000 trees. Helper: `tune_gbm` in `src/xpts/gbm.py`.

Caveats: the inner model sees one season and the final model two, so the chosen number of trees may be slightly too low;
and the inner validation of the 2017-18 fold (2016-17) is still before the label change, so tuning cannot prepare for it.

In [5]:
from xpts.gbm import tune_gbm

grid = [dict(num_leaves=l, min_child_samples=m) for l in (7, 15, 31, 63) for m in (20, 100, 500)]
chosen, rows_t = [], []
for season, tr, te in walk_forward(df["SEASON"], window=2):
    train_seasons = sorted(df.loc[tr, "SEASON"].unique())
    params = {}  # defaults when there is no season to validate on
    if len(train_seasons) == 2:
        inner_tr = (df["SEASON"] == train_seasons[0]).to_numpy()
        inner_val = (df["SEASON"] == train_seasons[1]).to_numpy()
        best = tune_gbm(df[inner_tr], y[inner_tr], df[inner_val], y[inner_val], grid).iloc[0]
        params = dict(num_leaves=int(best["num_leaves"]), min_child_samples=int(best["min_child_samples"]),
                      n_estimators=int(best["n_trees"]))
        chosen.append({"season": season, **params, "val_log_loss": best["val_log_loss"]})
    p = GBMModel(**params).fit(df[tr], y[tr]).predict_proba(df[te])[:, 1]
    oos.loc[df.index[te], "gbm_w2_tuned"] = p
    rows_t.append({"model": "gbm_w2_tuned", "season": season, **evaluate(y[te], p)})

display(pd.DataFrame(chosen).set_index("season").round(4))
res_t = pd.concat([res_w, pd.DataFrame(rows_t)])
order_t = ["logit_w1", "gbm_w2", "gbm_w2_tuned"]
display(res_t.groupby("model")[["log_loss", "brier", "auc"]].mean().loc[order_t].round(4))
display(res_t.pivot(index="season", columns="model", values="log_loss")[order_t].round(4))

,num_leaves,min_child_samples,n_estimators,val_log_loss
season,,,,
2016-17,7,100,137,0.6356
2017-18,31,20,107,0.6369
2018-19,7,500,135,0.6481
2019-20,31,100,98,0.6365
2020-21,31,20,117,0.6360
2021-22,15,100,72,0.6447
2022-23,15,20,113,0.6391
2023-24,31,20,101,0.6396
2024-25,31,20,136,0.6401


,log_loss,brier,auc
model,,,
logit_w1,0.6402,0.2255,0.6562
gbm_w2,0.6412,0.2260,0.6541
gbm_w2_tuned,0.6410,0.2259,0.6549


model,logit_w1,gbm_w2,gbm_w2_tuned
season,,,
2015-16,0.6367,0.6368,0.6368
2016-17,0.6386,0.6379,0.6381
2017-18,0.6478,0.6586,0.6561
2018-19,0.6372,0.6378,0.6385
2019-20,0.6367,0.6360,0.6359
2020-21,0.6446,0.6454,0.6451
2021-22,0.6395,0.6396,0.6396
2022-23,0.6401,0.6398,0.6397
2023-24,0.6406,0.6402,0.6401


In [6]:
# paired bootstrap: tuned vs logit and tuned vs untuned w2
ci_t = pd.DataFrame({(f"tuned - {ref}", period): paired_bootstrap(oos.loc[mask, "y"], oos.loc[mask, "gbm_w2_tuned"],
                                                                  oos.loc[mask, ref], games[mask])
                     for ref in ["logit_w1", "gbm_w2"]
                     for period, mask in [("all", oos["SEASON"].notna()), ("2018-19+", after)]}).T
display(ci_t.round(5))

diff   ci_low  ci_high  share_a_better
tuned - logit_w1 all       0.00077  0.00067  0.00087           0.000
                 2018-19+ -0.00004 -0.00014  0.00006           0.801
tuned - gbm_w2   all      -0.00022 -0.00027 -0.00017           1.000
                 2018-19+  0.00001 -0.00005  0.00007           0.369

**Observations.**
- Tuning gives almost nothing: mean log loss 0.6412 -> 0.6410 (tuned − untuned: −0.0002 [−0.00027; −0.00017]), and
  after 2018-19 the difference is zero. Against the logit after 2018-19 it is still a tie (−0.00004 [−0.00014; 0.00006]).
- The chosen parameters jump between folds (7-31 leaves, 72-137 trees): the validation surface is flat, like the
  hyperparameters of the baselines in Stage 3.
- Two very different models (spline logit, boosted trees) reach the same level with and without tuning. The limit is the
  **information in the features**, not the model. So the next step adds new information instead of a bigger model, and we
  keep the default parameters.

## 3c. Game-context features from play-by-play

**Data.** Play-by-play of all games (`scripts/00_data/07_fetch_pbp.py`, archive of stats.nba.com events) joined to the
shots by `GAME_ID` + event number (over 99.7% of shots matched in every season). `scripts/02_features/06_build_pbp_features.py`
builds four features, each from events **before** the shot only:
- `prev_event`: how the previous play ended (made shot, defensive / offensive rebound, steal, dead-ball turnover, free throw,
  foul, timeout, ...), substitutions skipped;
- `secs_since_prev`: seconds from that event to the shot (fast break and putback indicator);
- `score_margin`: shooter's team score minus opponent score before the shot;
- `is_home`: shooter plays at home.

**Leakage check.** Nothing after the shot is used (no assist, block, shooting foul or the score after the shot). The score
margin is shifted by one event, because the score on a made shot's own row already contains that shot.
Timestamps are entered by hand with about 1-2 s precision, so `secs_since_prev` is noisy.

**Comparison.** LightGBM, 2-season window, default parameters: without context (`gbm_w2`), with all four features
(`gbm_w2_ctx`), and two ablations (only previous event + time, only home + margin). The logit has no context features,
so `gbm_w2_ctx` vs `logit_w1` compares model + information, not the models alone.

In [7]:
from xpts.gbm import CONTEXT_FEATURES

KEY = ["GAME_ID", "GAME_EVENT_ID"]
dfc = df.merge(pd.read_parquet("../data/processed/shots_context.parquet"), on=KEY, how="left", validate="1:1")
assert len(dfc) == len(df)
print(f"shots with context: {dfc['prev_event'].notna().mean():.4f}")
display(dfc.groupby("prev_event", observed=True)["SHOT_MADE_FLAG"].agg(n="size", fg_pct="mean")
        .sort_values("n", ascending=False).round(3))

feature_sets = {"gbm_w2_ctx": GBM_FEATURES + CONTEXT_FEATURES,
                "gbm_w2_prev": GBM_FEATURES + ["prev_event", "secs_since_prev"],
                "gbm_w2_home": GBM_FEATURES + ["is_home", "score_margin"]}
rows_c = []
for name, feats in feature_sets.items():
    for season, tr, te in walk_forward(dfc["SEASON"], window=2):
        p = GBMModel(features=feats).fit(dfc[tr], y[tr]).predict_proba(dfc[te])[:, 1]
        oos.loc[dfc.index[te], name] = p
        rows_c.append({"model": name, "season": season, **evaluate(y[te], p)})

res_c = pd.concat([res_w, pd.DataFrame(rows_c)])
order_c = ["logit_w1", "gbm_w2", *feature_sets]
display(res_c.groupby("model")[["log_loss", "brier", "auc"]].mean().loc[order_c].round(4))
display(res_c.pivot(index="season", columns="model", values="log_loss")[order_c].round(4))

shots with context: 0.9997


,n,fg_pct
prev_event,,
made_shot,666879,0.445
def_rebound,666102,0.464
off_rebound,248857,0.499
free_throw,183348,0.438
steal,143833,0.568
dead_turnover,115693,0.440
foul,115274,0.433
timeout,96586,0.427
period_start,29589,0.434


,log_loss,brier,auc
model,,,
logit_w1,0.6402,0.2255,0.6562
gbm_w2,0.6412,0.2260,0.6541
gbm_w2_ctx,0.6390,0.2251,0.6580
gbm_w2_prev,0.6390,0.2251,0.6580
gbm_w2_home,0.6411,0.2259,0.6545


model,logit_w1,gbm_w2,gbm_w2_ctx,gbm_w2_prev,gbm_w2_home
season,,,,,
2015-16,0.6367,0.6368,0.6323,0.6324,0.6366
2016-17,0.6386,0.6379,0.6340,0.6341,0.6378
2017-18,0.6478,0.6586,0.6539,0.6540,0.6575
2018-19,0.6372,0.6378,0.6359,0.6360,0.6377
2019-20,0.6367,0.6360,0.6357,0.6355,0.6360
2020-21,0.6446,0.6454,0.6433,0.6434,0.6453
2021-22,0.6395,0.6396,0.6379,0.6379,0.6396
2022-23,0.6401,0.6398,0.6385,0.6385,0.6397
2023-24,0.6406,0.6402,0.6391,0.6391,0.6402


In [8]:
# paired bootstrap: context vs no context, and context vs the logit
pairs = [("gbm_w2_ctx", "gbm_w2"), ("gbm_w2_prev", "gbm_w2"), ("gbm_w2_home", "gbm_w2"), ("gbm_w2_ctx", "logit_w1")]
ci_c = pd.DataFrame({(f"{a} - {b}", period): paired_bootstrap(oos.loc[mask, "y"], oos.loc[mask, a], oos.loc[mask, b],
                                                             games[mask])
                     for a, b in pairs
                     for period, mask in [("all", oos["SEASON"].notna()), ("2018-19+", after)]}).T
display(ci_c.round(5))
ci_cs = pd.DataFrame({s: paired_bootstrap(g["y"], g["gbm_w2_ctx"], g["logit_w1"], games[g.index])
                      for s, g in oos.groupby("SEASON")}).T
display(ci_cs.round(5))

diff   ci_low  ci_high  share_a_better
gbm_w2_ctx - gbm_w2   all      -0.00226 -0.00236 -0.00216           1.000
                      2018-19+ -0.00136 -0.00147 -0.00126           1.000
gbm_w2_prev - gbm_w2  all      -0.00225 -0.00235 -0.00215           1.000
                      2018-19+ -0.00138 -0.00148 -0.00128           1.000
gbm_w2_home - gbm_w2  all      -0.00016 -0.00021 -0.00012           1.000
                      2018-19+ -0.00004 -0.00008  0.00001           0.926
gbm_w2_ctx - logit_w1 all      -0.00127 -0.00140 -0.00113           1.000
                      2018-19+ -0.00142 -0.00155 -0.00128           1.000

,diff,ci_low,ci_high,share_a_better
2015-16,-0.00434,-0.00481,-0.00388,1.0
2016-17,-0.00462,-0.00507,-0.00417,1.0
2017-18,0.00610,0.00555,0.00664,0.0
2018-19,-0.00124,-0.00163,-0.00087,1.0
2019-20,-0.00101,-0.00144,-0.00059,1.0
2020-21,-0.00133,-0.00171,-0.00095,1.0
2021-22,-0.00155,-0.00189,-0.00121,1.0
2022-23,-0.00157,-0.00189,-0.00124,1.0
2023-24,-0.00147,-0.00180,-0.00115,1.0
2024-25,-0.00166,-0.00197,-0.00136,1.0


**Observations.**
- Context is worth about ten times more than tuning: `gbm_w2_ctx` vs `gbm_w2` is −0.0023 [−0.0024; −0.0022] over all seasons
  and −0.0014 after 2018-19. Mean log loss 0.6390, AUC 0.658.
- Almost all of it comes from the previous event and the time since it (`gbm_w2_prev` = `gbm_w2_ctx`): shots right after a
  steal go in 57% of the time, after an offensive rebound 50%, after a timeout 43%. Home court and score margin add only
  −0.00016, and nothing after 2018-19.
- The gain is larger in 2015-16 and 2016-17 (−0.004) than later (−0.001 to −0.002). Before the label change the
  `ACTION_TYPE` vocabulary was poorer, so the context replaced information that the newer labels (e.g. "Running", "Driving",
  "Putback") partly carry themselves.
- With context LightGBM beats `logit_w1` in 9 of 10 seasons (−0.0013 [−0.0014; −0.0011]); it is still worse in 2017-18.
  This is not yet a fair comparison of the models, because the logit has no context features.
- Next notebook (`06_main_model.ipynb`): the logit with the same context features, calibration and SHAP.